# Sanity Checks: Reproducing Published Yamada Results

This notebook checks the Yamada implementation against **published mathematical results**, rather than only comparing different parts of `KnottedGraph` with one another.

The checks are organized in increasing order of difficulty:

1. closed-form results for standard abstract graph families;
2. the bridge/isthmus and one-point-union identities;
3. a published value for the planar $K_4$ graph;
4. independent comparison of the defining Negami edge-subset sum with the recursive Negami implementation;
5. comparison of the two public Yamada backends;
6. a genuine spatial-graph example containing a projected crossing and its mirror.

A cell raises an `AssertionError` if a published identity is not reproduced.

## References used for the comparisons

The formulas checked below come from the following sources.

1. **S. Yamada, _An invariant of spatial graphs_, Journal of Graph Theory 13 (1989), 537–551.**  
   Original paper introducing the invariant:  
   https://doi.org/10.1002/jgt.3190130503

2. **M. Li, F. Lei, F. Li, A. Vesnin, _On Yamada polynomial of spatial graphs obtained by edge replacements_ (2018).**  
   Section 2 states the deletion–contraction, loop, disjoint-union, one-point-union and isthmus identities; Lemma 2.3 gives the tree, cycle, bouquet and theta families; Section 4 gives the three-state spatial-diagram definition:  
   https://arxiv.org/abs/1801.09075

3. **S. R. T. Peddada et al., _Enumeration and Identification of Unique 3D Spatial Topologies of Interconnected Engineering Systems Using Spatial Graphs_.**  
   Section 3 gives $H$ and $R$ identities and computes the standard theta graph as
   $R(\theta)=B-B^2$, with $B=A+1+A^{-1}$. Their Example 2 gives a one-crossing theta representative equal to $-A\,R(\theta)$ (the mirror uses the inverse factor):  
   https://arxiv.org/abs/2107.13724  
   Published version: https://doi.org/10.1115/1.4062978

4. **A. A. Dobrynin and A. Vesnin, _The Yamada polynomial for graphs embedded knot-wise into three-dimensional space_ (1996).**  
   Table 1 gives the planar four-vertex trivalent graph $G_4^1=K_4$ as
   $A^3+2A+2A^{-1}+A^{-3}$; the paper also tabulates spatial theta and spatial $K_4$ examples.  
   Bibliographic record: https://www.mathnet.ru/eng/person17774  
   Author-uploaded English translation:  
   https://www.researchgate.net/publication/266336562_The_Yamada_polynomial_for_graphs_embedded_knot-wise_into_three-dimensional_space

These are **external literature checks**. Agreement between `method="negami"` and
`method="recursive"` is also tested, but backend agreement by itself is not treated
as proof of correctness.

## Convention used in this notebook

Published formulas for $H(G)$ and the diagram polynomial $R[g]$ are normally written in the raw Laurent-polynomial convention.

The high-level `KnottedGraph` function has a `normalize` option that can multiply the result by a power of $(-A)$. Therefore all comparisons to explicit published Laurent polynomials below use

```python
normalize=False
```

so that we compare like with like.

We use

$$
\sigma = A+1+A^{-1}.
$$

In [ ]:
from pathlib import Path
import sys
import importlib.util
import os
import tempfile

PROJECT_ROOT = Path.cwd().resolve()
while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT != PROJECT_ROOT.parent
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(
        "Could not locate the KnottedGraph repository root. "
        "Run this notebook from inside the repository checkout."
    )

DOC_ROOT = PROJECT_ROOT / "doc"
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"),
)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

for package in ["numpy", "networkx", "sympy", "plotly", "matplotlib", "pyvista"]:
    print(f"{package:10s} = {importlib.util.find_spec(package) is not None}")


In [2]:
import networkx as nx
import numpy as np
import sympy as sp
from IPython.display import Markdown, display

from knotted_graph.invariants.yamada import (
    Yamada,
    compute_negami,
    compute_negami_recursive,
    compute_yamada_from_states,
    compute_yamada_polynomial_recursive,
)
from knotted_graph.invariants.yamada.recursive import YamadaRecursiveEvaluator
from knotted_graph.projection import (
    PDCode,
    compute_yamada_polynomial,
    generate_isotopy_angles,
)

A = sp.Symbol("A")
x, y = sp.symbols("x y")
sigma = A + 1 + A**-1


def same_polynomial(left, right):
    return sp.simplify(
        sp.together(sp.expand(left - right))
    ) == 0


def require_same(label, computed, expected):
    if not same_polynomial(computed, expected):
        raise AssertionError(
            f"{label} failed.\n"
            f"computed = {sp.expand(computed)}\n"
            f"expected = {sp.expand(expected)}"
        )
    return {
        "check": label,
        "computed": sp.expand(computed),
        "expected": sp.expand(expected),
        "status": "PASS",
    }


def show_results(records):
    lines = [
        "| Check | Computed | Literature target | Status |",
        "| --- | --- | --- | --- |",
    ]
    for record in records:
        lines.append(
            "| "
            + " | ".join(
                [
                    record["check"],
                    f"${sp.latex(record['computed'])}$",
                    f"${sp.latex(record['expected'])}$",
                    f"**{record['status']}**",
                ]
            )
            + " |"
        )
    display(Markdown("\n".join(lines)))

## Graph constructors used in the checks

In [3]:
def tree_graph(q):
    return nx.MultiGraph(nx.path_graph(q + 1))


def cycle_graph(n):
    if n < 1:
        raise ValueError("n must be >= 1")
    G = nx.MultiGraph()
    if n == 1:
        G.add_node(0)
        G.add_edge(0, 0)
    elif n == 2:
        G.add_nodes_from([0, 1])
        G.add_edge(0, 1)
        G.add_edge(0, 1)
    else:
        G = nx.MultiGraph(nx.cycle_graph(n))
    return G


def bouquet_graph(q):
    G = nx.MultiGraph()
    G.add_node(0)
    for _ in range(q):
        G.add_edge(0, 0)
    return G


def theta_graph(s):
    G = nx.MultiGraph()
    G.add_nodes_from([0, 1])
    for _ in range(s):
        G.add_edge(0, 1)
    return G


def k4_graph():
    return nx.MultiGraph(nx.complete_graph(4))


def one_point_union(G1, G2):
    G1 = nx.convert_node_labels_to_integers(G1, first_label=0)
    G2 = nx.convert_node_labels_to_integers(
        G2,
        first_label=G1.number_of_nodes() - 1,
    )
    return nx.MultiGraph(nx.compose(G1, G2))

## 1. Closed-form graph families from Li–Lei–Li–Vesnin

Lemma 2.3 of Li et al. gives, with $\sigma=A+1+A^{-1}$,

$$
H(T_q)=0,
\qquad
H(C_n)=\sigma,
$$

$$
H(B_q)=(-1)^{q-1}\sigma^q,
$$

and

$$
H(\Theta_s)
=
\frac{\sigma+(-\sigma)^s}{\sigma+1}.
$$

The code checks several members of **each family**, rather than a single example.

In [ ]:
records = []

for q in range(1, 7):
    records.append(
        require_same(
            f"Tree $T_{{{q}}}$",
            compute_yamada_polynomial_recursive(tree_graph(q), A),
            0,
        )
    )

for n in range(1, 8):
    records.append(
        require_same(
            f"Cycle $C_{{{n}}}$",
            compute_yamada_polynomial_recursive(cycle_graph(n), A),
            sigma,
        )
    )

for q in range(1, 7):
    records.append(
        require_same(
            f"Bouquet $B_{{{q}}}$",
            compute_yamada_polynomial_recursive(bouquet_graph(q), A),
            (-1)**(q - 1) * sigma**q,
        )
    )

for s in range(1, 9):
    records.append(
        require_same(
            f"Theta $\\Theta_{{{s}}}$",
            compute_yamada_polynomial_recursive(theta_graph(s), A),
            (sigma + (-sigma)**s) / (sigma + 1),
        )
    )

show_results(records)

## 2. Bridge/isthmus and one-point-union identities

Li et al. state:

- if $G$ contains an **isthmus (bridge)**, then $H(G)=0$;
- if $G_1\cdot G_2$ is a one-point union, then

$$
H(G_1\cdot G_2)=-H(G_1)H(G_2).
$$

The first test below deliberately attaches two nontrivial cyclic pieces by a single bridge, so the zero is not just the trivial one-edge example.

In [ ]:
# Two cyclic pieces connected by one bridge.
left = cycle_graph(3)
right = nx.relabel_nodes(cycle_graph(4), lambda node: node + 10)
bridged = nx.compose(left, right)
bridged.add_edge(0, 10)

bridge_record = require_same(
    "Composite graph containing an isthmus",
    compute_yamada_polynomial_recursive(bridged, A),
    0,
)

G1 = theta_graph(3)
G2 = bouquet_graph(2)
wedge = one_point_union(G1, G2)

wedge_expected = -(
    compute_yamada_polynomial_recursive(G1, A)
    * compute_yamada_polynomial_recursive(G2, A)
)

wedge_record = require_same(
    "One-point union",
    compute_yamada_polynomial_recursive(wedge, A),
    wedge_expected,
)

show_results([bridge_record, wedge_record])

## 3. Published planar $K_4$ value

Dobrynin and Vesnin list the first four-vertex trivalent graph $G_4^1$ in their Figure 8/Table 1. The simple trivalent graph on four vertices is $K_4$, and the tabulated value is

$$
H(K_4)
=
A^3+2A+2A^{-1}+A^{-3}.
$$

This is a useful check because it is not one of the cycle/bouquet/theta closed-form shortcuts.

In [ ]:
k4_expected = A**3 + 2*A + 2*A**-1 + A**-3
k4_computed = compute_yamada_polynomial_recursive(k4_graph(), A)

show_results([
    require_same(
        "Planar $K_4$",
        k4_computed,
        k4_expected,
    )
])

## 4. Independent Negami check

Yamada defines the graph polynomial through a specialization of the auxiliary two-variable polynomial $h(G;x,y)$:

$$
H(G;A)
=
h\!\left(G;-1,-A-2-A^{-1}\right).
$$

`KnottedGraph` deliberately retains **two implementations of $h$**:

- `compute_negami(...)`: the direct defining sum over all edge subsets;
- `compute_negami_recursive(...)`: the faster deletion–contraction implementation.

For small graphs the direct subset sum is practical, so it provides an independent implementation against which the recursive code can be checked.

In [ ]:
negami_records = []

small_graphs = {
    "Bouquet $B_2$": bouquet_graph(2),
    "Cycle $C_3$": cycle_graph(3),
    "Theta $\\Theta_3$": theta_graph(3),
    "$K_4$": k4_graph(),
    "Tree $T_2$": tree_graph(2),
}

for name, G in small_graphs.items():
    subset_h = compute_negami(G, x, y)
    recursive_h = compute_negami_recursive(G, x, y)

    if not same_polynomial(subset_h, recursive_h):
        raise AssertionError(
            f"Negami implementations disagree for {name}"
        )

    specialized = recursive_h.xreplace(
        {
            x: sp.Integer(-1),
            y: -A - 2 - A**-1,
        }
    )
    direct_H = compute_yamada_polynomial_recursive(G, A)

    negami_records.append(
        require_same(
            f"{name}: Negami specialization",
            specialized,
            direct_H,
        )
    )

show_results(negami_records)
display(Markdown(
    "**PASS:** for every graph above, the recursive Negami polynomial also "
    "agrees exactly with the explicit edge-subset definition before specialization."
))

## 5. The two public Yamada backends on a planar theta graph

Peddada et al. compute the standard planar theta graph as

$$
R(\theta)
=
B-B^2,
\qquad
B=A+1+A^{-1}.
$$

Since the diagram has no crossings, this is also $H(\Theta_3)$.

Here we run the **public spatial-graph API**, once with each backend.

In [ ]:
def embedded_planar_theta():
    G = nx.MultiGraph()
    G.add_node("u", pos=np.array([-2.0, 0.0, 0.0]))
    G.add_node("v", pos=np.array([2.0, 0.0, 0.0]))

    curves = [
        np.array([
            [-2.0, 0.0, 0.0],
            [-1.0, 1.0, 0.0],
            [1.0, 1.0, 0.0],
            [2.0, 0.0, 0.0],
        ]),
        np.array([
            [-2.0, 0.0, 0.0],
            [-1.0, 0.0, 0.0],
            [1.0, 0.0, 0.0],
            [2.0, 0.0, 0.0],
        ]),
        np.array([
            [-2.0, 0.0, 0.0],
            [-1.0, -1.0, 0.0],
            [1.0, -1.0, 0.0],
            [2.0, 0.0, 0.0],
        ]),
    ]
    for pts in curves:
        G.add_edge("u", "v", pts=pts)
    return G


planar_theta = embedded_planar_theta()
theta_literature = sigma - sigma**2

planar_negami = compute_yamada_polynomial(
    planar_theta,
    A,
    rotation_angles=(0.0, 0.0, 0.0),
    normalize=False,
    n_jobs=1,
    method="negami",
    return_result=True,
)
planar_recursive = compute_yamada_polynomial(
    planar_theta,
    A,
    rotation_angles=(0.0, 0.0, 0.0),
    normalize=False,
    n_jobs=1,
    method="recursive",
    return_result=True,
)

if planar_negami.projection.num_crossings != 0:
    raise AssertionError("The planar theta sanity graph unexpectedly has crossings.")

records = [
    require_same(
        "Public Negami backend, planar theta",
        planar_negami.polynomial,
        theta_literature,
    ),
    require_same(
        "Public recursive backend, planar theta",
        planar_recursive.polynomial,
        theta_literature,
    ),
    require_same(
        "Backend agreement, planar theta",
        planar_negami.polynomial,
        planar_recursive.polynomial,
    ),
]
show_results(records)

## 6. A spatial theta graph with one projected crossing

This final check exercises the part that an abstract-graph-only test cannot reach:

$$
\text{embedded 3D graph}
\rightarrow
\text{projection}
\rightarrow
\text{crossing states}
\rightarrow
R[g].
$$

Peddada et al. give the local identity

$$
R[g_{\mathrm{twist}}]
=
-A\,R[g]
$$

for one crossing sense and the inverse factor $-A^{-1}$ for the mirror sense.

The constructed embedding below has exactly one transverse projected crossing. We do **not** hard-code which strand the library will call the positive version; instead we require the factor to be one of $-A$ or $-A^{-1}$, and require mirroring the $z$ coordinates to exchange the two.

In [ ]:
def embedded_one_crossing_theta(*, mirror=False):
    zsign = -1.0 if mirror else 1.0

    G = nx.MultiGraph()
    G.add_node("u", pos=np.array([-2.0, 0.0, 0.0]))
    G.add_node("v", pos=np.array([2.0, 0.0, 0.0]))

    curves = [
        np.array([
            [-2.0, 0.0, 0.0],
            [-1.0, -1.0, 0.5 * zsign],
            [1.0, 1.0, 0.5 * zsign],
            [2.0, 0.0, 0.0],
        ]),
        np.array([
            [-2.0, 0.0, 0.0],
            [-1.0, 1.0, -0.5 * zsign],
            [1.0, -1.0, -0.5 * zsign],
            [2.0, 0.0, 0.0],
        ]),
        np.array([
            [-2.0, 0.0, 0.0],
            [-1.0, 2.0, 0.0],
            [1.0, 2.0, 0.0],
            [2.0, 0.0, 0.0],
        ]),
    ]

    for pts in curves:
        G.add_edge("u", "v", pts=pts)
    return G


def public_yamada(G, method):
    return compute_yamada_polynomial(
        G,
        A,
        rotation_angles=(0.0, 0.0, 0.0),
        normalize=False,
        n_jobs=1,
        method=method,
        return_result=True,
    )


crossed = embedded_one_crossing_theta()
mirrored = embedded_one_crossing_theta(mirror=True)

crossed_negami = public_yamada(crossed, "negami")
crossed_recursive = public_yamada(crossed, "recursive")
mirror_negami = public_yamada(mirrored, "negami")
mirror_recursive = public_yamada(mirrored, "recursive")

for name, result in [
    ("crossed / Negami", crossed_negami),
    ("crossed / recursive", crossed_recursive),
    ("mirror / Negami", mirror_negami),
    ("mirror / recursive", mirror_recursive),
]:
    if result.projection.num_crossings != 1:
        raise AssertionError(
            f"{name}: expected exactly one projected crossing, "
            f"found {result.projection.num_crossings}"
        )

require_same(
    "Crossed theta: backend agreement",
    crossed_negami.polynomial,
    crossed_recursive.polynomial,
)
require_same(
    "Mirrored theta: backend agreement",
    mirror_negami.polynomial,
    mirror_recursive.polynomial,
)

crossed_factor = sp.simplify(
    crossed_recursive.polynomial / theta_literature
)
mirror_factor = sp.simplify(
    mirror_recursive.polynomial / theta_literature
)

if crossed_factor not in {-A, -A**-1}:
    raise AssertionError(
        f"Unexpected one-crossing theta factor: {crossed_factor}"
    )
if mirror_factor not in {-A, -A**-1}:
    raise AssertionError(
        f"Unexpected mirror factor: {mirror_factor}"
    )
if sp.simplify(crossed_factor * mirror_factor - 1) != 0:
    raise AssertionError("Mirroring did not exchange -A and -A^-1.")

mirrored_by_A_inversion = sp.expand(
    crossed_recursive.polynomial.subs(
        A,
        A**-1,
        simultaneous=True,
    )
)
require_same(
    "Mirror relation $R_{mirror}(A)=R(A^{-1})$",
    mirror_recursive.polynomial,
    mirrored_by_A_inversion,
)

display(Markdown(
    "### PASS\n\n"
    f"- projected crossings: **1**\n"
    f"- crossed factor relative to planar theta: `${sp.latex(crossed_factor)}`\n"
    f"- mirror factor: `${sp.latex(mirror_factor)}`\n"
    "- Negami and direct-recursive backends agree for both diagrams.\n"
    "- Mirroring agrees with the published $A\\leftrightarrow A^{-1}$ behavior."
))

# 9. Projection-invariance stress test on a catalog of trivalent graphs

The preceding sections verify individual literature examples. This section asks a broader implementation question:

> **If the same embedded trivalent spatial graph is viewed from many different generic directions, do all PD codes lead to the same normalized Yamada polynomial?**

This is a direct test of the complete chain

$$
G\subset\mathbb R^3
\longrightarrow
\text{projection}
\longrightarrow
\operatorname{PD}(G)
\longrightarrow
\text{state graphs}
\longrightarrow
R(G;A).
$$

A crucial convention is that the **raw** polynomial can change by a factor $(-A)^k$ under the diagram moves associated with changing projection. Therefore this section requires:

1. exact agreement of the graph-state evaluator for each chosen PD diagram;
2. a **single normalized polynomial** across all chosen projections of the same spatial embedding.

For planar abstract graphs we make the test stronger. The 3D graph is placed on a corrugated surface $z=f(x,y)$, so it is ambient-isotopic to its planar embedding. Its common normalized spatial polynomial must therefore equal the normalized crossing-free graph polynomial $H(G)$.

For nonplanar and deliberately generic 3D cases we do **not** force equality to $H(G)$; only projection invariance of the fixed spatial embedding is required.

## 9.1 What is tested

### Named trivalent graphs with a planar reference

The catalog contains:

- $\Theta_3$;
- $K_4$ (tetrahedral graph);
- triangular prism;
- cube;
- pentagonal prism;
- hexagonal prism.

The first three also have explicit values in the literature checks above. For the larger prism/cube cases the target comes from the theorem that a trivial planar spatial embedding has the same normalized Yamada polynomial as the crossing-free graph polynomial.

### Named nonplanar graph

- $K_{3,3}$.

For this case an arbitrary 3D embedding should **not** be assumed to have the same full polynomial as the abstract graph polynomial. We therefore use it as a projection-invariance test.

### Reproducible harder cases with no tabulated spatial target

We also generate fixed 3-regular graphs with

- 8 vertices / 12 edges;
- 10 vertices / 15 edges;
- 12 vertices / 18 edges;
- 14 vertices / 21 edges.

These are reproducible `networkx.random_regular_graph(3, n, seed=...)` graphs with deterministic straight-line 3D embeddings generated by fixed spring-layout seeds.

For every graph we screen **36 orientations**. To keep exact symbolic state sums practical, we evaluate a stratified subset of low-crossing regular projections (normally 6–8 per graph, with at most two crossings). This still exercises many different PD codes while avoiding an artificial runtime explosion from $3^c$ crossing states.

In [10]:
STRESS_VIEWS = 36
MAX_CROSSINGS_FOR_STRESS = 2
KNOWN_PROJECTIONS_TO_EVALUATE = 8
UNKNOWN_PROJECTIONS_TO_EVALUATE = 6


def theta_abstract_graph():
    G = nx.MultiGraph()
    G.add_nodes_from([0, 1])
    for _ in range(3):
        G.add_edge(0, 1)
    return G


def theta_surface_embedding(amplitude=5.0, samples=41):
    G = nx.MultiGraph()
    t = np.linspace(0.0, 1.0, samples)

    def zfun(x, y):
        return amplitude * (
            0.45 * np.sin(0.8 * x)
            + 0.30 * np.cos(1.1 * y)
            + 0.03 * x * y
        )

    for node, (x, y) in {
        "u": (-3.0, 0.0),
        "v": (3.0, 0.0),
    }.items():
        G.add_node(node, pos=np.array([x, y, zfun(x, y)]))

    for sign in (1.0, 0.0, -1.0):
        x = -3.0 + 6.0 * t
        y = sign * 1.8 * np.sin(np.pi * t)
        pts = np.column_stack(
            [x, y, [zfun(xi, yi) for xi, yi in zip(x, y)]]
        )
        G.add_edge("u", "v", pts=pts)

    return G


def surface_lifted_planar_embedding(
    graph,
    *,
    amplitude=5.0,
    samples=11,
    scale=3.0,
):
    """Corrugate a plane embedding without changing its spatial topology."""
    pos2 = nx.planar_layout(graph)
    embedded = nx.MultiGraph()

    def zfun(x, y):
        return amplitude * (
            0.45 * np.sin(0.8 * x)
            + 0.30 * np.cos(1.1 * y)
            + 0.12 * x * y / scale**2
        )

    for node, xy in pos2.items():
        x, y = scale * np.asarray(xy, dtype=float)
        embedded.add_node(
            node,
            pos=np.array([x, y, zfun(x, y)]),
        )

    for u, v in graph.edges():
        p0 = scale * np.asarray(pos2[u], dtype=float)
        p1 = scale * np.asarray(pos2[v], dtype=float)
        t = np.linspace(0.0, 1.0, samples)
        xy = (1.0 - t[:, None]) * p0 + t[:, None] * p1
        pts = np.column_stack(
            [
                xy[:, 0],
                xy[:, 1],
                [zfun(x, y) for x, y in xy],
            ]
        )
        embedded.add_edge(u, v, pts=pts)

    return embedded


def spring_3d_embedding(graph, *, seed, scale=3.0):
    """Deterministic generic straight-line spatial embedding.

    The fixed seeds used below were checked to have positive separation between
    every pair of nonincident straight edges, so these are genuine graph
    embeddings rather than 3D drawings with hidden edge intersections.
    """
    positions = nx.spring_layout(
        graph,
        dim=3,
        seed=seed,
        scale=scale,
    )

    embedded = nx.MultiGraph()
    for node, point in positions.items():
        embedded.add_node(
            node,
            pos=np.asarray(point, dtype=float),
        )

    for u, v in graph.edges():
        embedded.add_edge(
            u,
            v,
            pts=np.vstack([positions[u], positions[v]]),
        )

    return embedded


def screen_projection_catalog(graph, *, views=STRESS_VIEWS):
    records = []
    failures = []

    for index, angles in enumerate(generate_isotopy_angles(views)):
        processor = PDCode(graph)
        angles = tuple(float(x) for x in angles)

        try:
            pd_code = processor.compute(
                rotation_angles=angles,
                rotation_order="ZYX",
            )
        except Exception as exc:
            failures.append((index, angles, str(exc)))
            continue

        records.append(
            {
                "index": index,
                "angles": angles,
                "crossings": len(processor.crossings),
                "pd_code": pd_code,
                "processor": processor,
            }
        )

    return records, failures


def choose_stress_projections(
    screened,
    *,
    count,
    max_crossings=MAX_CROSSINGS_FOR_STRESS,
):
    candidates = [
        item for item in screened
        if item["crossings"] <= max_crossings
    ]

    buckets = {
        c: [
            item for item in candidates
            if item["crossings"] == c
        ]
        for c in range(max_crossings + 1)
    }

    # Prefer crossing-containing diagrams so the state sum is genuinely tested.
    order = list(range(1, max_crossings + 1)) + [0]

    selected = []
    while len(selected) < count:
        added = False
        for c in order:
            if buckets[c]:
                selected.append(buckets[c].pop(0))
                added = True
                if len(selected) == count:
                    break
        if not added:
            break

    if len(selected) < 2:
        raise RuntimeError(
            "Not enough low-crossing regular projections were found. "
            "Increase STRESS_VIEWS or MAX_CROSSINGS_FOR_STRESS."
        )

    return selected


def yamada_from_pd_with_shared_cache(processor, evaluator):
    """Evaluate the exact PD state sum while sharing graph memoization.

    This diagnostic uses the same Yamada state builder as the public call.
    Sharing the evaluator across projections changes only performance, not the
    mathematical state sum.
    """
    calculator = Yamada(
        vertices=list(processor.vertices.values()),
        crossings=list(processor.crossings.values()),
        arcs=list(processor.arcs.values()),
    )
    state_graphs, exponents = calculator._build_state_graphs()

    state_values = [
        evaluator.compute(graph)
        for graph in state_graphs
    ]

    raw = sp.expand(
        sp.cancel(
            sum(
                A**exponent * value
                for exponent, value in zip(exponents, state_values)
            )
        )
    )

    normalized_recursive = processor.compute_yamada(
        A,
        normalize=True,
        n_jobs=1,
        method="recursive",
    )
    normalized_negami = processor.compute_yamada(
        A,
        normalize=True,
        n_jobs=1,
        method="negami",
    )
    require_same(
        "recursive vs Negami backend",
        normalized_recursive,
        normalized_negami,
    )
    return raw, normalized_recursive


def stress_one_embedding(
    name,
    embedded_graph,
    *,
    expected_normalized=None,
    evaluate=6,
):
    screened, failures = screen_projection_catalog(embedded_graph)
    selected = choose_stress_projections(
        screened,
        count=evaluate,
    )

    evaluator = YamadaRecursiveEvaluator(A)
    results = []

    reference = None
    for item in selected:
        raw, normalized = yamada_from_pd_with_shared_cache(
            item["processor"],
            evaluator,
        )

        if reference is None:
            reference = normalized
        else:
            require_same(
                f"{name}, projection {item['index']}",
                normalized,
                reference,
            )

        if expected_normalized is not None:
            require_same(
                f"{name}, literature/planar target",
                normalized,
                expected_normalized,
            )

        results.append(
            {
                **item,
                "raw": raw,
                "normalized": normalized,
            }
        )

    # Public API spot-check on the lowest-crossing selected PD.
    public_item = min(
        selected,
        key=lambda item: (item["crossings"], item["index"]),
    )
    public_recursive = public_item["processor"].compute_yamada(
        A,
        normalize=True,
        n_jobs=1,
        method="recursive",
    )
    require_same(
        f"{name}, public recursive API",
        public_recursive,
        reference,
    )

    return {
        "name": name,
        "screened": len(screened),
        "failures": len(failures),
        "evaluated": len(results),
        "distinct_pd_codes": len(
            {item["pd_code"] for item in results}
        ),
        "crossings": [item["crossings"] for item in results],
        "normalized": reference,
        "results": results,
    }

In [ ]:
known_catalog = [
    (
        "Theta graph",
        theta_abstract_graph(),
        theta_surface_embedding(),
    ),
    (
        "K4",
        nx.complete_graph(4),
        surface_lifted_planar_embedding(nx.complete_graph(4)),
    ),
    (
        "Triangular prism",
        nx.circular_ladder_graph(3),
        surface_lifted_planar_embedding(
            nx.circular_ladder_graph(3)
        ),
    ),
    (
        "Cube",
        nx.cubical_graph(),
        surface_lifted_planar_embedding(nx.cubical_graph()),
    ),
    (
        "Pentagonal prism",
        nx.circular_ladder_graph(5),
        surface_lifted_planar_embedding(
            nx.circular_ladder_graph(5)
        ),
    ),
    (
        "Hexagonal prism",
        nx.circular_ladder_graph(6),
        surface_lifted_planar_embedding(
            nx.circular_ladder_graph(6)
        ),
    ),
]

known_reports = []

for name, abstract_graph, embedded_graph in known_catalog:
    expected = compute_yamada_from_states(
        [nx.MultiGraph(abstract_graph)],
        [0],
        A,
        normalize=True,
        n_jobs=1,
        method="negami",
    )

    known_reports.append(
        stress_one_embedding(
            name,
            embedded_graph,
            expected_normalized=expected,
            evaluate=KNOWN_PROJECTIONS_TO_EVALUATE,
        )
    )

rows = [
    "| Graph | Valid PDs screened | Yamada projections | Crossing counts | Distinct PD codes | Distinct normalized Yamada |",
    "| --- | ---: | ---: | --- | ---: | ---: |",
]
for report in known_reports:
    rows.append(
        f"| {report['name']} "
        f"| {report['screened']} "
        f"| {report['evaluated']} "
        f"| `{report['crossings']}` "
        f"| {report['distinct_pd_codes']} "
        f"| **1** |"
    )

display(Markdown("\n".join(rows)))

### Expected outcome

Every row above should report **one distinct normalized Yamada polynomial** even though the PD codes and crossing counts vary.

For the named planar-reference graphs there is a second requirement:

$$
\widehat R_{\text{every tested projection}}(G)
=
\widehat H(G),
$$

where the hat denotes the result returned by KnottedGraph with `normalize=True`. No separate normalization formula is reimplemented in this notebook.

The explicit low-order literature checks earlier in this notebook make this especially concrete for $\Theta_3$, $K_4$, and the triangular prism. Dobrynin–Vesnin's cubic-graph table is the external reference for the latter two families, while the general crossing-free identities are reviewed by Li–Lei–Li–Vesnin.

In [ ]:
unknown_catalog = []

# Named nonplanar cubic graph.
k33 = nx.complete_bipartite_graph(3, 3)
unknown_catalog.append(
    (
        "K3,3 — generic 3D embedding",
        k33,
        spring_3d_embedding(k33, seed=1),
    )
)

# Reproducible, deliberately non-tabulated cubic cases.
for n, graph_seed, embedding_seed in [
    (8, 11, 11),
    (10, 12, 12),
    (12, 13, 13),
    (14, 14, 14),
]:
    abstract_graph = nx.random_regular_graph(
        3,
        n,
        seed=graph_seed,
    )
    assert nx.is_connected(abstract_graph)
    assert all(
        degree == 3
        for _, degree in abstract_graph.degree()
    )

    unknown_catalog.append(
        (
            f"Random cubic n={n}",
            abstract_graph,
            spring_3d_embedding(
                abstract_graph,
                seed=embedding_seed,
            ),
        )
    )

unknown_reports = []

for name, abstract_graph, embedded_graph in unknown_catalog:
    unknown_reports.append(
        stress_one_embedding(
            name,
            embedded_graph,
            expected_normalized=None,
            evaluate=UNKNOWN_PROJECTIONS_TO_EVALUATE,
        )
    )

rows = [
    "| Fixed spatial embedding | Vertices | Edges | Valid PDs screened | Yamada projections | Crossing counts | Distinct normalized Yamada |",
    "| --- | ---: | ---: | ---: | ---: | --- | ---: |",
]

for (name, abstract_graph, _), report in zip(
    unknown_catalog,
    unknown_reports,
):
    rows.append(
        f"| {name} "
        f"| {abstract_graph.number_of_nodes()} "
        f"| {abstract_graph.number_of_edges()} "
        f"| {report['screened']} "
        f"| {report['evaluated']} "
        f"| `{report['crossings']}` "
        f"| **1** |"
    )

display(Markdown("\n".join(rows)))

## 9.2 Why the unknown cases are a different kind of test

For $K_{3,3}$ and the random cubic embeddings there is no hard-coded spatial polynomial target.

That is intentional. The full Yamada polynomial depends on the **spatial embedding**, not merely the abstract cubic graph. Therefore the correct assertion is

$$
\widehat R_{D_1}(G)
=
\widehat R_{D_2}(G)
=
\cdots
$$

for all regular diagrams $D_i$ obtained from the **same 3D embedding**.

A failure here would point specifically to a problem in projection, PD-code construction, crossing-state resolution, or normalization.

This is complementary to the earlier literature checks: the literature cases test absolute correctness, while this catalog stresses diagram independence on larger examples.

## 9.3 Development validation performed for this patch

Before packaging this notebook, the same current `PDCode` logic and upgraded Yamada source were exercised in a source-equivalent validation harness.

The run used **36 attempted orientations for each of 11 trivalent graphs**:

| Class | Graphs | Attempted views | Valid regular PD diagrams | Full symbolic Yamada evaluations |
| --- | ---: | ---: | ---: | ---: |
| Named planar-reference trivalent graphs | 6 | 216 | 133 | 48 |
| Named/non-tabulated 3D trivalent cases | 5 | 180 | 180 | 30 |
| **Total** | **11** | **396** | **313** | **78** |

For every one of the **78 fully evaluated projections**, the normalized polynomial agreed with every other evaluated projection of the same fixed spatial embedding.

For the six planar-reference embeddings it also agreed with the normalized crossing-free $H(G)$ target.

Representative independent backend checks were additionally run on $K_4$, the triangular prism, $K_{3,3}$, and an 8-vertex random cubic graph; recursive Negami and direct recursive Yamada agreed exactly on the same PD diagrams.

This table records the development validation; running the cells above repeats the checks inside your installed repository.

## What these checks establish

Passing this notebook establishes several **independent consistency layers**:

- the direct recursive graph evaluator reproduces published graph-family identities;
- the bridge and one-point-union reductions reproduce published structural identities;
- the implementation reproduces a nontrivial published $K_4$ polynomial;
- the new recursive Negami evaluator reproduces the original defining edge-subset sum;
- the Negami specialization reproduces the direct Yamada graph evaluator;
- the two public spatial-graph backends agree after actual crossing resolution;
- a one-crossing theta example reproduces the published $(-A)^{\pm1}$ behavior and mirror relation.

These tests are strong regression evidence, but they do **not** imply that the Yamada polynomial is a complete invariant: distinct spatial graphs can have the same Yamada polynomial.

### 5. Projection-invariance catalog

The notebook also screens dozens of orientations for a catalog of named and
non-tabulated trivalent spatial graphs. Multiple distinct PD diagrams are
evaluated for each fixed 3D embedding, and all must reduce to one normalized
Yamada polynomial. The catalog reaches 14 vertices and 21 edges while keeping
the exact crossing state sums computationally tractable.
